<a href="https://colab.research.google.com/github/allengr220/cognitive-ledger/blob/main/longevity/nhanes_physiorisk_ablation_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PhysioRisk Ablation (Model Comparison)

## Dataset
NHANES 1999–2018 linked mortality

## Outcome
- Time: PERMTH_INT
- Event: MORTSTAT

## Purpose
Compare PhysioRisk models against standard survival models to evaluate:

- dependence on chronological age
- preservation of physiological signal
- contribution of CRP

## Models included

### Standard models
- BaselineOnly (age spline + sex)
- RawBiomarkers (biomarkers only)
- AgePlusBiomarkers (age + biomarkers)

### PhysioRisk models
- PhysioRisk_v1 (3-axis)
- PhysioRisk_v2 (3-axis + linear CRP)
- PhysioRisk_v3 (3-axis + CRP interactions)

### Combined models
- TotalRisk_v1
- TotalRisk_v2
- TotalRisk_v3

## Temporal split
- Train: 1999–2010
- Test: 2011–2018

## Key evaluation metrics
- C-index
- Hazard ratio per 1 SD
- Correlation with age

## Goal
Demonstrate that PhysioRisk captures mortality-relevant physiological variation that is not reducible to chronological age.

## Notes
- Uses two cohorts:
  - full biomarker cohort (v1)
  - CRP-complete cohort (v2/v3)
- Results interpreted within cohort context

In [ ]:
!pip install lifelines

In [ ]:
#setup
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import SplineTransformer
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

# === Paths ===
DATA_PATH = "/content/drive/MyDrive/LRE/data/nhanes_1999_2018_extended_mortality_model_dataset.parquet"
OUTDIR = "/content/drive/MyDrive/LRE/data/pub"

# === Saved scored outputs ===
V1_TRAIN_PATH = f"{OUTDIR}/physiorisk_v1_train.parquet"
V1_TEST_PATH  = f"{OUTDIR}/physiorisk_v1_test.parquet"

V2_TRAIN_PATH = f"{OUTDIR}/physiorisk_3axis_crp_v2_train_scored.parquet"
V2_TEST_PATH  = f"{OUTDIR}/physiorisk_3axis_crp_v2_test_scored.parquet"

V3_TRAIN_PATH = f"{OUTDIR}/physiorisk_3axis_crp_interactions_v3_train_scored.parquet"
V3_TEST_PATH  = f"{OUTDIR}/physiorisk_3axis_crp_interactions_v3_test_scored.parquet"

# === Core variables ===
TIME_COL = "PERMTH_INT"
EVENT_COL = "MORTSTAT"
AGE_COL = "RIDAGEYR"
SEX_COL = "RIAGENDR"

# === Biomarkers ===
BIOMARKERS = [
    "BMXBMI",
    "BMXWAIST",
    "BPXSY1",
    "BPXDI1",
    "LBXGH",
    "LBDHDD",
    "LBXTC",
    "LBXTR",
    "LBXSAL",
    "LBXSCR",
    "LBXSAPSI",
    "LBXWBCSI",
    "LBXLYPCT",
    "LBXGLU",
    "LBXMCV",
    "LBXRDW",
]

N_AGE_KNOTS = 5
EPS = 1e-8

In [ ]:
#load saved model outputs
v1_train = pd.read_parquet(V1_TRAIN_PATH).copy()
v1_test  = pd.read_parquet(V1_TEST_PATH).copy()

v2_train = pd.read_parquet(V2_TRAIN_PATH).copy()
v2_test  = pd.read_parquet(V2_TEST_PATH).copy()

v3_train = pd.read_parquet(V3_TRAIN_PATH).copy()
v3_test  = pd.read_parquet(V3_TEST_PATH).copy()

print("v1:", v1_train.shape, v1_test.shape)
print("v2:", v2_train.shape, v2_test.shape)
print("v3:", v3_train.shape, v3_test.shape)

v1: (13090, 8) (6739, 8)
v2: (13086, 8) (2124, 8)
v3: (13086, 8) (2124, 8)


In [ ]:
# Cell 3 — load source dataset and define BOTH cohorts

df = pd.read_parquet(DATA_PATH).copy()

# derive cycle_start_year if needed
if "cycle_start_year" not in df.columns:
    if "CYCLE" in df.columns:
        df["cycle_start_year"] = df["CYCLE"].astype(str).str.extract(r"(\d{4})")[0].astype(int)
    elif "SDDSRVYR" in df.columns:
        cycle_map = {
            1: 1999, 2: 2001, 3: 2003, 4: 2005, 5: 2007,
            6: 2009, 7: 2011, 8: 2013, 9: 2015, 10: 2017
        }
        df["cycle_start_year"] = df["SDDSRVYR"].map(cycle_map)
    else:
        raise ValueError("Need cycle_start_year, CYCLE, or SDDSRVYR")

# harmonize time column if needed
if TIME_COL not in df.columns:
    if "PERMTH_EXM" in df.columns and TIME_COL == "PERMTH_INT":
        df[TIME_COL] = df["PERMTH_EXM"]
    else:
        raise ValueError(f"{TIME_COL} not found in source dataset")

# cohort 1: no-CRP cohort (matches v1)
required_v1 = [TIME_COL, EVENT_COL, AGE_COL, SEX_COL] + BIOMARKERS + ["cycle_start_year"]
df_v1 = df.dropna(subset=required_v1).copy()

train_df_v1 = df_v1[df_v1["cycle_start_year"] <= 2009].copy()
test_df_v1  = df_v1[df_v1["cycle_start_year"] >= 2011].copy()

# cohort 2: CRP-complete cohort (matches v2/v3)
required_crp = [TIME_COL, EVENT_COL, AGE_COL, SEX_COL] + BIOMARKERS + ["LBXHSCRP", "cycle_start_year"]
df_crp = df.dropna(subset=required_crp).copy()

train_df_crp = df_crp[df_crp["cycle_start_year"] <= 2009].copy()
test_df_crp  = df_crp[df_crp["cycle_start_year"] >= 2011].copy()

print("v1 cohort:", train_df_v1.shape, test_df_v1.shape)
print("CRP cohort:", train_df_crp.shape, test_df_crp.shape)

v1 cohort: (13035, 33) (6705, 33)
CRP cohort: (13031, 33) (2114, 33)


In [ ]:
#helper functions
def make_age_spline(train_age, test_age, n_knots=5):
    spl = SplineTransformer(n_knots=n_knots, degree=3, include_bias=False)
    tr = spl.fit_transform(train_age.to_numpy().reshape(-1, 1))
    te = spl.transform(test_age.to_numpy().reshape(-1, 1))

    cols = [f"age_spline_{i}" for i in range(tr.shape[1])]
    tr = pd.DataFrame(tr, index=train_age.index, columns=cols)
    te = pd.DataFrame(te, index=test_age.index, columns=cols)
    return tr, te


def hr_per_sd(df_scores, score_col):
    tmp = df_scores[[TIME_COL, EVENT_COL, score_col]].copy()
    tmp["score_z"] = (tmp[score_col] - tmp[score_col].mean()) / (tmp[score_col].std() + EPS)

    if tmp["score_z"].std() < 1e-6:
        return np.nan, np.nan, np.nan

    cph = CoxPHFitter(penalizer=0.1)
    cph.fit(tmp[[TIME_COL, EVENT_COL, "score_z"]], duration_col=TIME_COL, event_col=EVENT_COL)

    coef = cph.params_["score_z"]
    ci = cph.confidence_intervals_.loc["score_z"]

    hr = np.exp(coef)
    hr_lo = np.exp(ci.iloc[0])
    hr_hi = np.exp(ci.iloc[1])
    return hr, hr_lo, hr_hi


def evaluate_score(df_scores, score_col):
    cidx = concordance_index(
        df_scores[TIME_COL],
        -df_scores[score_col],
        df_scores[EVENT_COL]
    )
    hr, hr_lo, hr_hi = hr_per_sd(df_scores, score_col)
    corr_age = np.corrcoef(df_scores[score_col], df_scores[AGE_COL])[0, 1]

    return {
        "risk_score": score_col,
        "C_index": cidx,
        "HR_per_1SD": hr,
        "HR_95CI_low": hr_lo,
        "HR_95CI_high": hr_hi,
        "corr_age": corr_age,
    }


def append_metrics(rows, cohort_name, model_name, dfx, score_col):
    out = evaluate_score(dfx, score_col)
    out["cohort"] = cohort_name
    out["model"] = model_name
    rows.append(out)

In [ ]:
# Cell 5 — rebuild baseline-only model on v1 cohort

age_train_spl_v1, age_test_spl_v1 = make_age_spline(
    train_df_v1[AGE_COL], test_df_v1[AGE_COL], n_knots=N_AGE_KNOTS
)

X_train_base = pd.concat([train_df_v1[[TIME_COL, EVENT_COL, SEX_COL]], age_train_spl_v1], axis=1)
X_test_base  = pd.concat([test_df_v1[[TIME_COL, EVENT_COL, SEX_COL]], age_test_spl_v1], axis=1)

baseline_cph = CoxPHFitter(penalizer=0.01)
baseline_cph.fit(X_train_base, duration_col=TIME_COL, event_col=EVENT_COL)

baseline_train = baseline_cph.predict_log_partial_hazard(
    X_train_base.drop(columns=[TIME_COL, EVENT_COL])
).rename("BaselineOnly")

baseline_test = baseline_cph.predict_log_partial_hazard(
    X_test_base.drop(columns=[TIME_COL, EVENT_COL])
).rename("BaselineOnly")

baseline_train_df = train_df_v1[[TIME_COL, EVENT_COL, AGE_COL]].copy()
baseline_test_df  = test_df_v1[[TIME_COL, EVENT_COL, AGE_COL]].copy()
baseline_train_df["BaselineOnly"] = baseline_train
baseline_test_df["BaselineOnly"] = baseline_test

In [ ]:
# Cell 6 — raw biomarkers Cox on v1 cohort

X_train_raw = pd.concat([train_df_v1[[TIME_COL, EVENT_COL]], train_df_v1[BIOMARKERS]], axis=1)
X_test_raw  = pd.concat([test_df_v1[[TIME_COL, EVENT_COL]], test_df_v1[BIOMARKERS]], axis=1)

raw_cph = CoxPHFitter(penalizer=0.1)
raw_cph.fit(X_train_raw, duration_col=TIME_COL, event_col=EVENT_COL)

raw_train = raw_cph.predict_log_partial_hazard(
    X_train_raw.drop(columns=[TIME_COL, EVENT_COL])
).rename("RawBiomarkers")

raw_test = raw_cph.predict_log_partial_hazard(
    X_test_raw.drop(columns=[TIME_COL, EVENT_COL])
).rename("RawBiomarkers")

raw_train_df = train_df_v1[[TIME_COL, EVENT_COL, AGE_COL]].copy()
raw_test_df  = test_df_v1[[TIME_COL, EVENT_COL, AGE_COL]].copy()
raw_train_df["RawBiomarkers"] = raw_train
raw_test_df["RawBiomarkers"] = raw_test

In [ ]:
# Cell 7 — age + biomarkers Cox on v1 cohort

X_train_age_bio = pd.concat(
    [train_df_v1[[TIME_COL, EVENT_COL, SEX_COL]], age_train_spl_v1, train_df_v1[BIOMARKERS]],
    axis=1
)
X_test_age_bio = pd.concat(
    [test_df_v1[[TIME_COL, EVENT_COL, SEX_COL]], age_test_spl_v1, test_df_v1[BIOMARKERS]],
    axis=1
)

age_bio_cph = CoxPHFitter(penalizer=0.1)
age_bio_cph.fit(X_train_age_bio, duration_col=TIME_COL, event_col=EVENT_COL)

age_bio_train = age_bio_cph.predict_log_partial_hazard(
    X_train_age_bio.drop(columns=[TIME_COL, EVENT_COL])
).rename("AgePlusBiomarkers")

age_bio_test = age_bio_cph.predict_log_partial_hazard(
    X_test_age_bio.drop(columns=[TIME_COL, EVENT_COL])
).rename("AgePlusBiomarkers")

age_bio_train_df = train_df_v1[[TIME_COL, EVENT_COL, AGE_COL]].copy()
age_bio_test_df  = test_df_v1[[TIME_COL, EVENT_COL, AGE_COL]].copy()
age_bio_train_df["AgePlusBiomarkers"] = age_bio_train
age_bio_test_df["AgePlusBiomarkers"] = age_bio_test

In [ ]:
#inspect saved column names
print("v1 train cols:")
print(v1_train.columns.tolist())

print("\nv2 train cols:")
print(v2_train.columns.tolist())

print("\nv3 train cols:")
print(v3_train.columns.tolist())

v1 train cols:
['PERMTH_INT', 'MORTSTAT', 'RIDAGEYR', 'RIAGENDR', 'cycle_start_year', 'BaselineRisk', 'PhysioRisk', 'TotalRisk']

v2 train cols:
['PERMTH_INT', 'MORTSTAT', 'RIDAGEYR', 'RIAGENDR', 'cycle_start_year', 'BaselineRisk', 'PhysioRisk_v2', 'TotalRisk_v2']

v3 train cols:
['PERMTH_INT', 'MORTSTAT', 'RIDAGEYR', 'RIAGENDR', 'cycle_start_year', 'BaselineRisk', 'PhysioRisk_v3', 'TotalRisk_v3']


In [ ]:
rows = []

# v1-cohort rebuilt models
append_metrics(rows, "train_temporal", "BaselineOnly", baseline_train_df, "BaselineOnly")
append_metrics(rows, "test_temporal",  "BaselineOnly", baseline_test_df,  "BaselineOnly")

append_metrics(rows, "train_temporal", "RawBiomarkers", raw_train_df, "RawBiomarkers")
append_metrics(rows, "test_temporal",  "RawBiomarkers", raw_test_df,  "RawBiomarkers")

append_metrics(rows, "train_temporal", "AgePlusBiomarkers", age_bio_train_df, "AgePlusBiomarkers")
append_metrics(rows, "test_temporal",  "AgePlusBiomarkers", age_bio_test_df,  "AgePlusBiomarkers")

# saved v1
append_metrics(rows, "train_temporal", "PhysioRisk_v1", v1_train, "PhysioRisk")
append_metrics(rows, "test_temporal",  "PhysioRisk_v1", v1_test,  "PhysioRisk")
append_metrics(rows, "train_temporal", "TotalRisk_v1",  v1_train, "TotalRisk")
append_metrics(rows, "test_temporal",  "TotalRisk_v1",  v1_test,  "TotalRisk")

# saved v2 on CRP cohort
append_metrics(rows, "train_temporal", "PhysioRisk_v2", v2_train, "PhysioRisk_v2")
append_metrics(rows, "test_temporal",  "PhysioRisk_v2", v2_test,  "PhysioRisk_v2")
append_metrics(rows, "train_temporal", "TotalRisk_v2",  v2_train, "TotalRisk_v2")
append_metrics(rows, "test_temporal",  "TotalRisk_v2",  v2_test,  "TotalRisk_v2")

# saved v3 on CRP cohort
append_metrics(rows, "train_temporal", "PhysioRisk_v3", v3_train, "PhysioRisk_v3")
append_metrics(rows, "test_temporal",  "PhysioRisk_v3", v3_test,  "PhysioRisk_v3")
append_metrics(rows, "train_temporal", "TotalRisk_v3",  v3_train, "TotalRisk_v3")
append_metrics(rows, "test_temporal",  "TotalRisk_v3",  v3_test,  "TotalRisk_v3")

ablation_df = pd.DataFrame(rows)[[
    "cohort",
    "model",
    "risk_score",
    "C_index",
    "HR_per_1SD",
    "HR_95CI_low",
    "HR_95CI_high",
    "corr_age",
]]

display(ablation_df.sort_values(["cohort", "model"]).reset_index(drop=True))

,cohort,model,risk_score,C_index,HR_per_1SD,HR_95CI_low,HR_95CI_high,corr_age
0,test_temporal,AgePlusBiomarkers,AgePlusBiomarkers,0.838276,1.633471,1.541276,1.731180,9.162723e-01
1,test_temporal,BaselineOnly,BaselineOnly,0.823523,1.574021,1.482972,1.670659,9.781139e-01
2,test_temporal,PhysioRisk_v1,PhysioRisk,0.649695,1.258927,1.191172,1.330537,9.087818e-02
3,test_temporal,PhysioRisk_v2,PhysioRisk_v2,0.670364,1.228379,1.110998,1.358161,1.053185e-01
4,test_temporal,PhysioRisk_v3,PhysioRisk_v3,0.696657,1.288354,1.177861,1.409212,1.474329e-01
5,test_temporal,RawBiomarkers,RawBiomarkers,0.771147,1.444633,1.367710,1.525883,5.894393e-01
6,test_temporal,TotalRisk_v1,TotalRisk,0.841020,1.698798,1.602250,1.801163,9.572465e-01
7,test_temporal,TotalRisk_v2,TotalRisk_v2,0.833506,1.426015,1.271383,1.599455,9.564964e-01
8,test_temporal,TotalRisk_v3,TotalRisk_v3,0.839242,1.450921,1.294230,1.626582,9.448100e-01
9,train_temporal,AgePlusBiomarkers,AgePlusBiomarkers,0.866134,2.488635,2.412610,2.567056,9.155595e-01


In [ ]:
#pivoted paper-style tables
paper_cindex = ablation_df.pivot(index="model", columns="cohort", values="C_index")
paper_hr = ablation_df.pivot(index="model", columns="cohort", values="HR_per_1SD")
paper_corr = ablation_df.pivot(index="model", columns="cohort", values="corr_age")

print("C-index")
display(paper_cindex.sort_index())

print("HR per 1 SD")
display(paper_hr.sort_index())

print("corr(age)")
display(paper_corr.sort_index())

C-index


cohort,test_temporal,train_temporal
model,,
AgePlusBiomarkers,0.838276,0.866134
BaselineOnly,0.823523,0.847154
PhysioRisk_v1,0.649695,0.591341
PhysioRisk_v2,0.670364,0.604174
PhysioRisk_v3,0.696657,0.607285
RawBiomarkers,0.771147,0.815672
TotalRisk_v1,0.841020,0.865794
TotalRisk_v2,0.833506,0.866935
TotalRisk_v3,0.839242,0.865705


HR per 1 SD


cohort,test_temporal,train_temporal
model,,
AgePlusBiomarkers,1.633471,2.488635
BaselineOnly,1.574021,2.364433
PhysioRisk_v1,1.258927,1.225370
PhysioRisk_v2,1.228379,1.248308
PhysioRisk_v3,1.288354,1.253827
RawBiomarkers,1.444633,2.005577
TotalRisk_v1,1.698798,2.593333
TotalRisk_v2,1.426015,2.611619
TotalRisk_v3,1.450921,2.599290


corr(age)


cohort,test_temporal,train_temporal
model,,
AgePlusBiomarkers,0.916272,9.155595e-01
BaselineOnly,0.978114,9.794151e-01
PhysioRisk_v1,0.090878,2.722129e-16
PhysioRisk_v2,0.105319,-8.666367e-16
PhysioRisk_v3,0.147433,1.756156e-02
RawBiomarkers,0.589439,6.075085e-01
TotalRisk_v1,0.957246,9.589268e-01
TotalRisk_v2,0.956496,9.574314e-01
TotalRisk_v3,0.944810,9.602031e-01


In [ ]:
#save outputs
ablation_df.to_csv(f"{OUTDIR}/physiorisk_ablation_v1_metrics.csv", index=False)
paper_cindex.to_csv(f"{OUTDIR}/physiorisk_ablation_v1_cindex.csv")
paper_hr.to_csv(f"{OUTDIR}/physiorisk_ablation_v1_hr.csv")
paper_corr.to_csv(f"{OUTDIR}/physiorisk_ablation_v1_corr_age.csv")